# CV11 — Ghép điểm giữa hai ảnh

Mỗi source token phân loại vị trí token đích. Correlation [B,Nsrc,Ndst], target [B,Nsrc], -100 đánh dấu source không có nhãn. Hai ảnh ở đây giống nhau nên dùng ánh xạ identity.

Năm cell code, chạy lần lượt. Fixture không phải benchmark; xem REFERENCE.md để chọn dữ liệu thật và paper gốc.


## Cell 1 — Môi trường và seed

Dùng CPU để kiểm tra trước. Tìm repo root từ thư mục notebook, đặt seed và import PyTorch. Không tải pretrained weights.


In [ ]:
from pathlib import Path
import sys
import torch
from torch import nn
import torch.nn.functional as F

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'vision/curriculum').is_dir())
sys.path.insert(0, str(ROOT))
torch.manual_seed(42)
torch.set_num_threads(2)
device = torch.device('cpu')  # Change to cuda after the CPU example works.
print('Device:', device)


## Cell 2 — Tạo batch và model

Mỗi source token phân loại vị trí token đích. Correlation [B,Nsrc,Ndst], target [B,Nsrc], -100 đánh dấu source không có nhãn. Hai ảnh ở đây giống nhau nên dùng ánh xạ identity. Dữ liệu trong cell là fixture tổng hợp để bạn đọc được shape; thay bằng Dataset thật trước khi so metric.


In [ ]:
from vision.curriculum.cv11_matching import Matcher, loss_fn, soft_coordinates
first = torch.rand(2, 3, 32, 32, device=device)
second = first.clone()
model = Matcher(context=False).to(device)
with torch.no_grad():
    feature_height, feature_width = model.enc(first).shape[-2:]
labels = torch.arange(feature_height * feature_width, device=device).repeat(2, 1)
batch = (first, second, labels)


## Cell 3 — Forward và baseline

Đọc shape/output trước khi train. Các số in ra chỉ minh họa phép tính và luồng dự đoán.


In [ ]:
model.eval()
with torch.no_grad():
    correlations = model(first, second)
    coordinates = soft_coordinates(correlations, feature_height, feature_width)
print('Correlation:', correlations.shape)
print('Expected destination coordinates in feature pixels:', coordinates.shape)


## Cell 4 — Một bước cập nhật

Tính đúng loss của nhiệm vụ, backward và cập nhật tham số. Một bước không phải quy trình huấn luyện hoàn chỉnh. TENT tự thực hiện update bên trong adapt.


In [ ]:
model.train()
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=3e-4)
optimizer.zero_grad(set_to_none=True)
loss = loss_fn(model, *batch)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
print('One-step training loss:', loss.detach().item())


## Cell 5 — Chẩn đoán và hướng nâng cấp

Cosine features → supervised correlation → cross-attention → coarse-to-fine. Soft coordinate đang dùng pixel trên lưới feature, chưa phải pixel ảnh gốc.


In [ ]:
model.eval()
with torch.no_grad():
    correlations = model(first, second)
    identity_accuracy = (correlations.argmax(-1) == labels).float().mean()
print('Identical-image feature match accuracy:', identity_accuracy.item())
print('Upgrade: context=True; then coarse-to-fine refinement. Real metrics: PCK or homography error.')


## Thử nghiệm tiếp theo

Cosine features → supervised correlation → cross-attention → coarse-to-fine. Soft coordinate đang dùng pixel trên lưới feature, chưa phải pixel ảnh gốc.

Giữ nguyên split và ngân sách huấn luyện. Đọc [module](../cv11_matching.py) và [tài liệu gốc](../REFERENCE.md).
